In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

def generate_feature_impact_plot(csv_path, output_png):
    print(f"\n🌍 Loading Data: {csv_path}")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['Spearman'])
    df = df[df['Spearman'] > 0.05].copy()

    # Standardize regions
    df['Region'] = df['Subregions'].replace({
        'CD3_VH + CD3_VL': 'VH+VL', 'CD3_VH': 'VH', 'CD3_VL': 'VL',
        'CD3_VH + scFv': 'VH+scFv', 'CD3_VL + scFv': 'VL+scFv'
    }).astype(str).str.replace(' ', '')
    
    # 🌟 CHANGED: Reordered the columns exactly as requested
    region_order = ['scFv', 'VH+VL', 'VL', 'VH']

    # 1. Exact Token Extraction
    df['Feature_List'] = df['Features'].apply(lambda x: [t.strip() for t in str(x).split('+')])
    
    unique_features = set()
    for lst in df['Feature_List']:
        unique_features.update(lst)
    unique_features = {f for f in unique_features if f}

    # 2. Build the Exact 'With vs Without' Matrix
    impact_records = []
    for feat in unique_features:
        has_feat_mask = df['Feature_List'].apply(lambda lst: feat in lst)
        
        for reg in region_order:
            reg_mask = df['Region'] == reg
            
            with_scores = df[reg_mask & has_feat_mask]['Spearman']
            without_scores = df[reg_mask & (~has_feat_mask)]['Spearman']
            
            for s in with_scores:
                impact_records.append({'Feature': feat, 'Region': reg, 'Status': 'With', 'Spearman': s})
            for s in without_scores:
                impact_records.append({'Feature': feat, 'Region': reg, 'Status': 'Without', 'Spearman': s})
                
    df_impact = pd.DataFrame(impact_records)

    # 3. Smart Relevance Sorting (Global Lift across all regions)
    base_lifts = {}
    for feat in unique_features:
        sub_with = df_impact[(df_impact['Feature'] == feat) & (df_impact['Status'] == 'With')]['Spearman']
        sub_without = df_impact[(df_impact['Feature'] == feat) & (df_impact['Status'] == 'Without')]['Spearman']
        mean_with = sub_with.mean() if not sub_with.empty else 0
        mean_without = sub_without.mean() if not sub_without.empty else 0
        base_lifts[feat] = mean_with - mean_without

    family_lifts = {}
    for feat in unique_features:
        base_name = feat.replace('i-', '')
        if base_name not in family_lifts:
            family_members = [f for f in unique_features if f.replace('i-', '') == base_name]
            family_lifts[base_name] = np.mean([base_lifts[f] for f in family_members])

    sorted_families = sorted(family_lifts.keys(), key=lambda x: family_lifts[x], reverse=True)

    sorted_features = []
    for base in sorted_families:
        if base in unique_features:
            sorted_features.append(base)
        if f"i-{base}" in unique_features:
            sorted_features.append(f"i-{base}")

    # 4. Plotting the 4-Panel Dashboard
    sns.set_theme(style="whitegrid")
    
    fig_height = max(12, len(sorted_features) * 0.6)
    fig, axes = plt.subplots(1, 4, figsize=(24, fig_height), sharey=True, sharex=True)
    fig.suptitle("Exact Feature Impact: With vs. Without (By Structural Region)", 
                 fontsize=20, fontweight='bold', y=0.98)

    palette = {'With': '#2ca02c', 'Without': '#a1c9f4'}
    status_order = ['With', 'Without']

    for i, reg in enumerate(region_order):
        ax = axes[i]
        subset = df_impact[df_impact['Region'] == reg]
        
        sns.stripplot(data=subset, y='Feature', x='Spearman', hue='Status',
                      order=sorted_features, hue_order=status_order, palette=palette, 
                      dodge=True, alpha=0.15, size=3, zorder=1, ax=ax)
        
        sns.boxplot(data=subset, y='Feature', x='Spearman', hue='Status',
                    order=sorted_features, hue_order=status_order, palette=palette, ax=ax,
                    width=0.7, showfliers=False, linewidth=1.5, zorder=10,
                    showmeans=True, meanprops={"marker":"o", 
                                               "markerfacecolor":"white", 
                                               "markeredgecolor":"black",
                                               "markersize":"6"})
        
        ax.set_title(f"Region: {reg}", fontsize=16, fontweight='bold', pad=10)
        ax.set_xlabel("Cross-Validation Spearman Correlation", fontsize=12)
        
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        
        if i == 0:
            ax.set_ylabel("Evaluated Feature (Sorted by Global Impact)", fontsize=14)
        else:
            ax.set_ylabel("")

    custom_legend = [
        Patch(facecolor='#2ca02c', edgecolor='black', linewidth=1.5, label='With Feature'),
        Patch(facecolor='#a1c9f4', edgecolor='black', linewidth=1.5, label='Without Feature')
    ]
    fig.legend(handles=custom_legend, title="Inclusion Status", 
               loc='upper left', bbox_to_anchor=(0.01, 0.99), 
               fontsize=12, title_fontsize=13, framealpha=0.95, edgecolor='gray')

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 Feature Impact Dashboard saved to: {output_png}")

# Execute:
# generate_feature_impact_plot('model_comparison/global_exhaustive_search_checkpoint_XGBoost_HMW.csv', 'model_comparison/Feature_Impact_Dashboard.png')

In [50]:
generate_feature_impact_plot('XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv',
                             'model_comparison/Feature_Impact_Dashboard-8.png')


🌍 Loading Data: XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv
📊 Feature Impact Dashboard saved to: model_comparison/Feature_Impact_Dashboard-8.png


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

def generate_exact_parsimony_plot(csv_path, output_png, output_excel, threshold=0.55, exclude_regions=None):
    if exclude_regions is None:
        exclude_regions = []
        
    print(f"\n🌍 Loading Data: {csv_path}")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['Spearman'])
    df = df[df['Spearman'] > 0.05].copy()

    # Standardize regions
    df['Region'] = df['Subregions'].replace({
        'CD3_VH + CD3_VL': 'VH+VL', 'CD3_VH': 'VH', 'CD3_VL': 'VL',
        'CD3_VH + scFv': 'VH+scFv', 'CD3_VL + scFv': 'VL+scFv'
    }).astype(str).str.replace(' ', '')
    
    region_colors = {'VH': '#1f77b4', 'VL': '#ff7f0e', 'VH+VL': '#d62728', 'scFv': '#9467bd'}

    # 1. Exact Feature Count Champion Extraction
    champion_indices = df.groupby('Num_Features')['Spearman'].idxmax()
    df_champions = df.loc[champion_indices].copy().sort_values(by='Num_Features', ascending=True)
    
    # 🌟 NEW: Calculate the Smooth, Unfiltered Monotonic Pareto Front
    # This ratchet mechanism creates the smooth flatlining curve showing absolute max potential
    max_scores = df.groupby('Num_Features')['Spearman'].max().sort_index()
    pareto_front = max_scores.cummax()
    
    # 2. Partitioning logic for Visuals and Export
    mask_top = (df_champions['Spearman'] >= threshold) & (~df_champions['Region'].isin(exclude_regions))
    df_top = df_champions[mask_top]
    df_faded = df_champions[~mask_top]
    
    # Excel Export
    with pd.ExcelWriter(output_excel) as writer:
        df_champions.to_excel(writer, sheet_name='All_Champions', index=False)
        df_top.to_excel(writer, sheet_name='Top_Performers', index=False)
        
    print(f"💾 EXPORTED: {len(df_champions)} Total Champions | {len(df_top)} Highlighted Targets saved to {output_excel}")

    # 3. Plotting the Clean Dashboard
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Background: All Suboptimal Permutations
    sns.scatterplot(data=df, x='Num_Features', y='Spearman', color='lightgray', 
                    alpha=0.3, s=20, ax=ax, legend=False)

    # 🌟 NEW: Plot the smooth Monotonic Pareto Curve
    ax.plot(pareto_front.index, pareto_front.values, color='black', 
            linestyle='--', linewidth=2, zorder=4, alpha=0.7)

    # Plot FADED exact champions (Below threshold or Excluded Region)
    sns.scatterplot(data=df_faded, x='Num_Features', y='Spearman', hue='Region', 
                    palette=region_colors, s=50, alpha=0.3, edgecolor='none', 
                    zorder=6, ax=ax, legend=False)
                    
    # Plot HIGHLIGHTED exact champions (Above threshold AND Allowed Region)
    sns.scatterplot(data=df_top, x='Num_Features', y='Spearman', hue='Region', 
                    palette=region_colors, s=150, edgecolor='black', linewidth=1.5, 
                    zorder=10, ax=ax, legend=False)
                    
    # Target Threshold Line
    ax.axhline(y=threshold, color='red', linestyle=':', linewidth=2.5)

    # Dynamic Title
    title_extra = f" | Excluded from Highlights: {', '.join(exclude_regions)}" if exclude_regions else ""
    ax.set_title(f"Absolute Performance Ceiling per Exact Feature Count\n(True Champions: {len(df_champions)} | Highlighted Targets: {len(df_top)}{title_extra})", 
                 fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel("Total Number of Input Features", fontsize=12)
    ax.set_ylabel("Cross-Validation Spearman Correlation", fontsize=12)
    
    # Hardcoded Bulletproof Legend
    custom_legend = [
        Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Global Pareto Front (Smooth Ceiling)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#9467bd', markeredgecolor='black', markersize=10, label='scFv'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markeredgecolor='black', markersize=10, label='VH+VL'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markeredgecolor='black', markersize=10, label='VH'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e', markeredgecolor='black', markersize=10, label='VL'),
        Line2D([0], [0], color='red', linestyle=':', linewidth=2.5, label=f'Target Threshold ({threshold})')
    ]
    
    ax.legend(handles=custom_legend, title="Region & Architecture Limits", loc='lower right', 
              framealpha=0.95, edgecolor='gray', fontsize=11, title_fontsize=12)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 Parsimony Curve Dashboard saved to: {output_png}")

# Execute:
# generate_exact_parsimony_plot(
#     csv_path='model_comparison/global_exhaustive_search_checkpoint_XGBoost_HMW.csv', 
#     output_png='model_comparison/Exact_Parsimony_Curve_Filtered.png',
#     output_excel='model_comparison/Elite_Candidates_For_Inference.xlsx',
#     threshold=0.55,
#     exclude_regions=['VL']
# )

In [46]:
generate_exact_parsimony_plot(
    csv_path='XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv', 
    output_png='model_comparison/Exact_Parsimony_Curve-6.png',
    output_excel='model_comparison/Elite_Candidates_For_Inference-6.xlsx',
    threshold=0.55,
    exclude_regions=['VL']  # <-- Blocks these from the plot and export
)


🌍 Loading Data: XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv
💾 EXPORTED: 381 Total Champions | 19 Highlighted Targets saved to model_comparison/Elite_Candidates_For_Inference-5.xlsx
📊 Parsimony Curve Dashboard saved to: model_comparison/Exact_Parsimony_Curve-5.png


In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

def plot_standalone_pareto_front(csv_path, output_filename='model_comparison/Standalone_Pareto_Front.png'):
    print(f"\n🌍 Loading Data: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Clean the data
    df = df.dropna(subset=['Spearman'])
    df = df[df['Spearman'] > 0.05].copy()

    # Standardize regions
    df['Region'] = df['Subregions'].replace({
        'CD3_VH + CD3_VL': 'VH+VL', 
        'CD3_VH': 'VH', 
        'CD3_VL': 'VL',
        'CD3_VH + scFv': 'VH+scFv', 
        'CD3_VL + scFv': 'VL+scFv'
    }).astype(str).str.replace(' ', '')
    
    region_colors = {'VH': '#1f77b4', 'VL': '#ff7f0e', 'VH+VL': '#d62728', 'scFv': '#9467bd'}
    
    # Calculate the Monotonic Pareto Front (The Ratchet)
    # 1. Find the max score at every exact feature count
    max_scores = df.groupby('Num_Features')['Spearman'].max().sort_index()
    # 2. Use cummax() so the line flatlines until a model strictly beats the previous high score
    pareto_front = max_scores.cummax()

    # Setup the figure
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Scatter plot of all models (exactly matching the original styling)
    sns.scatterplot(data=df, x='Num_Features', y='Spearman', hue='Region', 
                    palette=region_colors, alpha=0.6, s=45, edgecolor=None, ax=ax)
    
    # Overlay the smooth Pareto Front
    ax.plot(pareto_front.index, pareto_front.values, color='black', 
            linestyle='--', linewidth=2, zorder=10)

    # Formatting
    ax.set_title("Complexity vs. Reward (The Pareto Front)\nMapping the Limits of Feature Dimensionality", 
                 fontsize=18, fontweight='bold', pad=15)
    ax.set_xlabel("Total Number of Input Features", fontsize=14)
    ax.set_ylabel("Out-of-Sample Spearman Correlation", fontsize=14)
    
    # Adjust axis limits slightly for breathing room
    ax.set_xlim(-50, df['Num_Features'].max() + 100)
    
    # Custom Legend
    custom_legend = [
        Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Global Pareto Front'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=10, label='VH'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e', markersize=10, label='VL'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markersize=10, label='VH+VL'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#9467bd', markersize=10, label='scFv')
    ]
    
    ax.legend(handles=custom_legend, title="Structural Region", loc='lower right', 
              framealpha=0.95, edgecolor='gray', fontsize=12, title_fontsize=13)

    plt.tight_layout()
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 Standalone Pareto plot saved to: {output_filename}")

# Execute:
# plot_standalone_pareto_front('model_comparison/global_exhaustive_search_checkpoint_XGBoost_HMW.csv')

In [48]:
plot_standalone_pareto_front('XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv')


🌍 Loading Data: XGBoost/global_exhaustive_search_checkpoint_XGBoost_SUBSET_50-50_HMW%.csv
📊 Standalone Pareto plot saved to: model_comparison/Standalone_Pareto_Front.png
